<a href="https://colab.research.google.com/github/jameshphan-png/Coding-Exercise---ML-Basics/blob/dev/Module_3_Coding_Exercise_Part_2_(Predict_Customer_Chrun).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ------------------------------------------------------------
# IMPORTANT (READ ME BEFORE STARTING)
#
# Ensure you have downloaded "churn_300.xlsx" as the excel file obtained from Github.
# USE THE EXCEL FILE AND UPLOAD INTO THE RUNTIME IN COLAB THROUGH THE 'Files' (ON LEFT SIDE BAR), AND DRAG AND DROP THERE.
# ------------------------------------------------------------

# ======================================================
# Dataset Source:
# Kaggle - "Churn Modelling" Dataset
# https://www.kaggle.com/datasets/manasvikirti/churn-modelling
# ======================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

# -----------------------------
# 1) Load data
# -----------------------------
INPUT_XLSX = "churn_300.xlsx"
df = pd.read_excel(INPUT_XLSX)

LABEL_COL = "Exited"  # 1 = churned, 0 = stayed

# -----------------------------
# 2) Ensure dataset has 300 rows + save new Excel
# -----------------------------
df_300 = df.sample(n=min(300, len(df)), random_state=42).reset_index(drop=True)

OUTPUT_XLSX = "/content/churn_300.xlsx"
df_300.to_excel(OUTPUT_XLSX, index=False)

print("\nDATASET INFO")
print("------------")
print(f"Using {len(df_300)} records in this dataset.")
print("Target column: Exited (1 = churned, 0 = stayed)")
print("Our goal is to analyze the likelihood that the customer may or may not come back, based on categorical data factors influencing churn.")

# -----------------------------
# 3) Define features/label for model
# -----------------------------
drop_cols_for_training = ["RowNumber", "CustomerId", "Surname"]
X = df_300.drop(columns=drop_cols_for_training + [LABEL_COL])
y = df_300[LABEL_COL]

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

# -----------------------------
# 4) Preprocessing
# -----------------------------
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder="drop"
)

# -----------------------------
# 5) Model: Logistic Regression
# -----------------------------
model = LogisticRegression(max_iter=2000)
clf = Pipeline(steps=[("preprocess", preprocess), ("model", model)])

# -----------------------------
# 6) Train/test split + training
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)
clf.fit(X_train, y_train)

# ======================================================
# 7) MODEL COEFFICIENTS (ADDED SECTION)
# ======================================================
# Logistic Regression learns one coefficient per processed feature.
# - Positive coefficient -> increases churn probability
# - Negative coefficient -> decreases churn probability
# Because we scaled numeric variables and one-hot encoded categories, the model is
# comparing features on a consistent scale.

print("\nMODEL COEFFICIENTS")
print("------------------")
print("Logistic Regression is based on the coefficients factoring from both StandardScaler & OneHotEncoder to translate columns and scaling them.")
print(
    "Explanation:\n"
    "- Each coefficient shows how that feature affects churn risk (scaled through both StandardScaler & OneHotEncoder).\n"
    "- Positive value -> Numbers attaining a high positive value will lead to higher churn probability - Influences lower retention\n"
    "- Negative value -> Numbers attaining a high negative value lower churn probability - Influences higher retention\n"
)

# Get feature names after preprocessing:
# Numeric names are unchanged; categorical expand into one-hot names
ohe = clf.named_steps["preprocess"].named_transformers_["cat"]
cat_feature_names = ohe.get_feature_names_out(cat_cols)

all_feature_names = np.concatenate([np.array(num_cols), cat_feature_names])

# Extract coefficients from the trained logistic regression model
coefficients = clf.named_steps["model"].coef_.ravel()  # shape: (n_features,)

coef_df = pd.DataFrame({
    "feature": all_feature_names,
    "coefficient": coefficients
})
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()

# Show the most influential features (top 15 by absolute magnitude)
top_n = 15
coef_top = coef_df.sort_values("abs_coefficient", ascending=False).head(top_n)

print(f"Top {top_n} most influential features (by absolute coefficient magnitudes):")
print(coef_top[["feature", "coefficient"]].to_string(index=False))

# Optional: save coefficients to Excel for submission
COEF_OUT_XLSX = "/content/logistic_regression_coefficients.xlsx"
coef_df.sort_values("abs_coefficient", ascending=False).to_excel(COEF_OUT_XLSX, index=False)
print(f"\n >A table of all coefficients (based on positive/negative magnitudes) is saved as: logistic_regression_coefficients.xlsx in the Files tab")

# ======================================================
# 8) FULL INSIGHT OUTPUT FOR ALL 300 RECORDS (WITH CHURN)
# ======================================================

all_proba = clf.predict_proba(X)[:, 1]
all_pred = (all_proba >= 0.5).astype(int)

insight_cols = [
    "RowNumber",
    "Surname",          # Username
    "CreditScore",
    "Geography",
    "Gender",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary",
    "Exited"
]

insight_cols_existing = [c for c in insight_cols if c in df_300.columns]

output_table = df_300[insight_cols_existing].copy()
output_table["churn_probability"] = all_proba
output_table["predicted_churn_0.5"] = all_pred

# -----------------------------
# 9) Save the output table to Excel
# -----------------------------

print("\nFULL PREDICTIONS TABLE (ALL 300 RECORDS)")
print("-------------------------------------------------------------------------------------")
print(
    "Explanation:\n"
    "- churn_probability column is the model’s estimated chance (0 to 1) that the customer will churn.\n"
    "- predicted_churn_0.5 column converts probability to a churn value using threshold 0.5:\n"
    "    If probability >= 0.5 → churn (1) - Customer most likely will NOT come back in the future\n"
    "    If probability <  0.5 → not churn (0) - Customer will most likely come back\n"
    "- The 'Exited' column is the ACTUAL churn label from the dataset (ground truth).\n"
)
OUT_PREDICTIONS_XLSX = "/content/churn_300_full_predictions.xlsx"
output_table.to_excel(OUT_PREDICTIONS_XLSX, index=False)
print(f">A full prediction chrun table is saved as: churn_300_full_predictions.xlsx in the files tab")
print(f"IMPORATNT: THIS CONTAINS THE ENTIRE OUTPUT OF ALL PREDICTED VALUES BASED ON CHURN. YOU WANT TO USE THIS FILE TO SEE THE OUTPUTTED RESULTS FOR CONVENIENCE PURPOSES.")

# -----------------------------
# 10) Business Explanation & Concepts
# -----------------------------
print(
    "\nBUSINESS USE\n"
    "------------\n"
    "Businesses can use churn probabilities to identify high-risk customers,\n"
    "prioritize retention efforts, and intervene early (offers/support) to reduce churn.\n"
    "It is important to acknowledge the churn probabilities that it impacts customer retentnion & performance for a company to continue running overall"
)

print("\nHOW BUSINESSES CAN USE THIS MODEL TO REDUCE CHURN")
print("------------------------------------------------")
print(
    "The churn probability produced by the logistic regression model represents\n"
    "the likelihood that a customer will leave the company.\n\n"

    "Businesses can use these predictions to:\n"
    "1) Identify high-risk customers (for example, customers with churn probability >= 0.5).\n"
    "2) Prioritize retention efforts such as discounts, personalized offers, or proactive support.\n"
    "3) Intervene early before customers actually leave, reducing revenue loss.\n"
    "4) Allocate resources more efficiently by focusing on customers most likely to churn.\n"
    "5) Evaluate on POSITIVE COEFFICIENTS that are highly influential to churn probability.\n\n"

    "By acting on churn predictions instead of waiting for customers to leave,\n"
    "companies can improve customer satisfaction, increase retention rates,\n"
    "and reduce the overall cost of acquiring new customers."
)


DATASET INFO
------------
Using 300 records in this dataset.
Target column: Exited (1 = churned, 0 = stayed)
Our goal is to analyze the likelihood that the customer may or may not come back, based on categorical data factors influencing churn.

MODEL COEFFICIENTS
------------------
Logistic Regression is based on the coefficients factoring from both StandardScaler & OneHotEncoder to translate columns and scaling them.
Explanation:
- Each coefficient shows how that feature affects churn risk (scaled through both StandardScaler & OneHotEncoder).
- Positive value -> Numbers attaining a high positive value will lead to higher churn probability - Influences lower retention
- Negative value -> Numbers attaining a high negative value lower churn probability - Influences higher retention

Top 15 most influential features (by absolute coefficient magnitudes):
          feature  coefficient
  Geography_Spain    -1.226312
Geography_Germany     0.847149
              Age     0.524541
   IsActiveM